In [ ]:
import os
import xarray as xr
import datetime as dt
import erddapy
from erddapy import ERDDAP
import pandas as pd
import matplotlib.dates as mdates
import numpy as np
import matplotlib.pyplot as plt
import cmocean
import matplotlib.dates as mdates
myFmtlong = mdates.DateFormatter('%m/%d\n%H:%M')
import datetime as dt
import bottleneck as bn
from gsw import SA_from_SP, CT_from_t, rho, p_from_z

## Set plotting parameters
SMALL_SIZE = 12
MEDIUM_SIZE = 15
BIGGER_SIZE = 20

# increase text sizes because the figure is so big
fac =1.5
plt.rc('font', size=SMALL_SIZE * fac)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE * fac)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE * fac)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE * fac)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE * fac *0.8)  # fontsize of the figure title

In [ ]:
## Paths -- edit these to match your local setup
DATA_DIR = './data'
FORCING_DIR = f'{DATA_DIR}/forcing'
OUTPUT_DIR = f'{DATA_DIR}/final_run/data'
FIGURES_DIR = f'{DATA_DIR}/final_run/figures'
IBTRACS_CSV = f'{DATA_DIR}/ibtracs.ALL.list.v04r00.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
save_prof19 = 1629339767 #8/19
save_prof28 =  1630114503 #8/28

In [ ]:
def distance_lat_lon_ddegrees(lat1,lon1,lat2,lon2):
    # Calculates distance between two sets of decimal degree coordinates in km
    R = 6373.0 #radius of the Earth [km]

    #define lat lon points (2 lat, 2 lon), convert to radians
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    
    #calculate the distance between both points
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    # Haversine formula - spherical trig (great circle distance)
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2 
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    #final distance is radius of the earth times c
    distance = R * c
     #return distance in km
    return distance

In [ ]:
# grab an individual gliders data
ioos_url = 'https://data.ioos.us/gliders/erddap'
def get_erddap_dataset(ds_id, variables=None, constraints=None, filetype=None):
    """
    Returns a netcdf dataset for a specified dataset ID (or dataframe if dataset cannot be converted to xarray)
    :param ds_id: dataset ID e.g. ng314-20200806T2040
    :param variables: optional list of variables
    :param constraints: optional list of constraints
    :param filetype: optional filetype to return, 'nc' (default) or 'dataframe'
    :return: netcdf dataset
    """
    #manually define each or they are set to none
    variables = variables or None
    constraints = constraints or None
    #define another filetype or default is netcdf - input is dataframe or nc
    filetype = filetype or 'nc'
    #make call to erddap server using tabledap, return netcdf
    e = ERDDAP(server=ioos_url,
               protocol='tabledap',
               response='nc')
    #set dataset id to ds_id (input to function)
    e.dataset_id = ds_id
    
    #if kwargs are defined, define as such
    if constraints:
        e.constraints = constraints
    if variables:
        e.variables = variables
    if filetype == 'nc':
        try:
            #conver to xarray dataset
            ds = e.to_xarray()
            #sort by time
            ds = ds.sortby(ds.time)
        except OSError:
            #if there isn't a dataset, ds is empty list
            print('No dataset available for specified constraints: {}'.format(ds_id))
            ds = []
        except TypeError:
            #convert to pandas and drop nans
            print('Cannot convert to xarray, providing dataframe: {}'.format(ds_id))
            ds = e.to_pandas().dropna()
    elif filetype == 'dataframe':
        #if filetype is dataframe, convert to pandas and drop nans
        ds = e.to_pandas().dropna()
    else:
        #if filetype is not recognized print message 
        print('Unrecognized filetype: {}. Needs to  be "nc" or "dataframe"'.format(filetype))
    # return dataset as xarray or pandas - whatever filetype was specified as
    return ds

def glider_dataset(gliderid, time_start=None, time_end=None, variables=None, filetype=None):
    """
    Return data from a specific glider
    """
    print('Retrieving glider dataset: {}'.format(gliderid))
    #define each in kwarg else it will be defined as none
    time_start = time_start or None
    time_end = time_end or None
    variables = variables or [
        'depth',
        'latitude',
        'longitude',
        'time',
        'precise_time',
        'temperature',
        'salinity',
        'density',
        'profile_id'
    ]
    # define filetype as dataframe or nc, default is nc
    filetype = filetype or 'nc'
    
    #create empty dictionary
    constraints = dict()
    #each if statement looking at kwarg, define as such
    if time_start:
        #convert datetime to string - decimal number
        constraints['time>='] = time_start.strftime('%Y-%m-%dT%H:%M:%SZ')
    if time_end:
        #convert datetime to string - decimal number
        constraints['time<='] = time_end.strftime('%Y-%m-%dT%H:%M:%SZ')
    #if one or no contraints given, will be defined as None
    if len(constraints) < 1:
        constraints = None
    #create empty dictionary
    kwargs = dict()
    #looking at kwargs and will define as such
    kwargs['variables'] = variables
    kwargs['constraints'] = constraints
    kwargs['filetype'] = filetype
    #Uses get_erddap_dataset function from above
    ds = get_erddap_dataset(gliderid, **kwargs)
    #checks if ds is an object and that it is a  pandas dataframe, else it is a xarray dataset
    if isinstance(ds, pd.core.frame.DataFrame):
        #loops through each column and splits (i.e. getting rid of units in the name)
        for col in ds.columns:
            ds.rename(columns={col: col.split(' ')[0]}, inplace=True)  # get rid of units in column names
        #converts each timestamp to string - decimal number
        ds['time'] = ds['time'].apply(lambda t: dt.datetime.strptime(t, '%Y-%m-%dT%H:%M:%SZ'))
    #return dataset - pandas or xarray
    return ds

In [ ]:
#define glider dataset
glider = 'ng645-20210613T0000'
#define export path
save_dir = DATA_DIR
#define start and end time
# t0 = dt.datetime(2021, 8, 25, 0, 0) #start
# t0 = dt.datetime(2021, 8, 28, 0, 0) #start
# t1 =  dt.datetime(2021, 9, 4, 0, 0) #end


t0 = dt.datetime(2021, 8, 17, 0, 0) #start
t1 =  dt.datetime(2021, 9, 1, 0, 0) #end
#define limits for plots?
ylims = [40, 1]
temp_xlim = [24, 31]
salinity_xlim = [34.75, 36]
density_xlim = [1021, 1025]
# ts_xlim = [34.25, 37.25]

# y_limits = [-20, 0]  # None
# c_limits = dict(temp=dict(shallow=np.arange(25, 30, .25)),
#                 salt=dict(shallow=np.arange(34, 37, .25)))


# initialize keyword arguments for glider functions
gargs = dict()
gargs['time_start'] = t0  # False
gargs['time_end'] =t1
gargs['filetype'] = 'dataframe'

#create folder for dataset
sdir_glider = os.path.join(save_dir, glider, 'transects', 'transect-ribbons')
os.makedirs(sdir_glider, exist_ok=True)
#grab glider dataset using function and kwargs defined above
glider_df = glider_dataset(glider, **gargs)

In [ ]:
#download data from website hyperlinked above
# Load Ida ibtracs data
ibtracs = pd.read_csv(IBTRACS_CSV, header=[0], skiprows=[1]) #header (columns), first row, drop skip second row
#subset to 2021 and hurricane Ida
ida = ibtracs[(ibtracs['SEASON']==2021) & (ibtracs['NAME']=='IDA')]
# reset index
ida = ida.reset_index()

In [ ]:
# Now I want to find the sea surface salinity of each profile to see if there might be a barrier layer there
#glider_df dataframe - dataset from single glider
glider_df
#sort dataframe by profile_id
prof_ids = np.unique(glider_df.profile_id.values)
#create empty lists
lat = []
lon = []
sss = []
datenum = []
ptime=[]
cnt=0
#loop through each profile id
for prof in prof_ids:
    # print(prof)
    # print(cnt)
    #create subset of each profile
    prof_df = glider_df[glider_df['profile_id'] == prof]
    #reset index
    prof_df = prof_df.reset_index()
    #grab first lat and lon value and append to respective list
    lat = np.append(lat, prof_df['latitude'].values[0])
    lon = np.append(lon, prof_df['longitude'].values[0])
    #grab first datetime
    datenum = np.append(datenum, mdates.date2num(prof_df['time'].values[0]))
    ptime=np.append(ptime,prof_df['precise_time'].values[0])
    #define depth
    _z = prof_df['depth'].values
    #create surface profile subset between 0 and 10m
    surf_prof_df = prof_df[(prof_df['depth']>0) & (prof_df['depth']<=10)] # May need to play with this range
    #reset index
    surf_prof_df = surf_prof_df.reset_index()
#     print(surf_prof_df)
    #append sea surface salinty
    sss = np.append(sss, np.nanmean(surf_prof_df['salinity'].values))
    # print(sss)
    cnt+=1

In [ ]:
save_profs=[save_prof19,save_prof28]

for prof in save_profs:
    save_prof=prof

    # Profile 1630015360 is 8/26 22:02:40.. closest to 8/27 00.. doesn't reach the surface so using profile before it.. also doing during 8/27 - 1630065375
    # curprof = 1630065375
    # curprof = 1630006364
    curprof=save_prof
    ind = prof_ids == curprof
    FWBL_prof_ids = prof_ids[ind]
    FWBL_dn = datenum[ind]
    FWBL_lat = lat[ind]
    FWBL_lon = lon[ind]
    count = 0
    #########################################################################################################
    # Pull profiles with low surface salinity
    _sal  = glider_df['salinity'][glider_df['profile_id'] == curprof].values
    _temp = glider_df['temperature'][glider_df['profile_id'] == curprof].values
    _z    = glider_df['depth'][glider_df['profile_id'] == curprof].values

    # if _z[0] > _z[-1]: # make sure we are alwasy looking from shallow to deep
    #     print('flipped profs')
    #     _sal  = _sal[::-1]
    #     _temp = _temp[::-1]
    #     _z    = _z[::-1]

    # Sort profiles by depth // shallow to deep
    _sal = _sal[np.argsort(_z)]
    _temp = _temp[np.argsort(_z)]
    _z = _z[np.argsort(_z)]

    ## Remove repeated depth values // there are only ~4 repeated depth values so this might be overkill
    uni = np.unique(_z)
    for num in uni:
        num_ind = np.where(_z==num)[0]
        tempmean = np.nanmean(_temp[num_ind])
        salmean = np.nanmean(_sal[num_ind])

        _z = np.delete(_z, num_ind[:-1])
        _temp = np.delete(_temp, num_ind[:-1])
        _sal = np.delete(_sal, num_ind[:-1])

        _sal[num_ind[0]] = salmean
        _temp[num_ind[0]] = tempmean
        _z[num_ind[0]] = num


    #########################################################################################################
    #########################################################################################################
    # Calculate ILD and MLD following Rudzin 2018 and deBruyner and Montague 2011
    thresh = 0.5
    # Calculate the initial rho profile (pressure approximated by depth in dbar, as in the original)
    SA = SA_from_SP(_sal, _z, FWBL_lon[count], FWBL_lat[count])
    CT = CT_from_t(SA, _temp, _z)
    ini_rho = rho(SA, CT, _z)
    # Calculate density at 2m depth
    ind_2m = np.argmin(np.abs(np.abs(_z) - 2))

    #     print(_z[ind_2m])

    SA_2m = SA_from_SP(_sal[ind_2m], 2, FWBL_lon[count], FWBL_lat[count])
    CT_2m = CT_from_t(SA_2m, _temp[ind_2m], 2)
    rho_2m = rho(SA_2m, CT_2m, 2)
    # Calculate the density if the temp was 0.5°C cooler
    CT_mld = CT_from_t(SA_2m, _temp[ind_2m] - thresh, 2)
    rho_mld = rho(SA_2m, CT_mld, 2)
    # Find where the initial rho profile == the mld rho
    mld = _z[np.argmin(np.abs(ini_rho-rho_mld))]
    # isothermal layer depth defined by Rudzin 2018
    ild = _z[(np.abs(_temp - _temp[0]) > thresh)][0] 
    #     print(mld)
    #     print(ild)

    if np.abs(_z[ind_2m] - 2) < 2: # make sure that the reference depth is actually near 2m
        #########################################################################################################
        # Calculate distance to track and distance to storm position at time of the profile
    #         _dist2idaKM = distance_lat_lon_ddegrees(ida['LAT'].values,ida['LON'].values, lat[prof_ids == curprof], lon[prof_ids == curprof]) #km
        _dist2idaKM = distance_lat_lon_ddegrees(ida['LAT'].values,ida['LON'].values, FWBL_lat[count], FWBL_lon[count]) #km

        # Calculate distance to the storm relative to the RMW 
        _dist2idaRMW = _dist2idaKM/(ida['USA_RMW'].values.astype(np.float) * 1.852) # convert rmw from nmile to km
        # Get the closest proximity to the storm
        _dist2trackKM = np.nanmin(_dist2idaKM) # in straight km from track
        _dist2trackRMW = np.nanmin(_dist2idaRMW) # relative to the RMW of the storm at each point
        #########################################################################################################

    print(_z[ind_2m])
    closeIC={
        "z": {"dims": ("z"), "data": _z},
        "sal": {"dims": ("z"), "data": _sal},
        "temp": {"dims": ("z"), "data": _temp},
        "datenum": {"dims": ("z"), "data": FWBL_dn[count] * np.ones_like(_z)},
        "profnum": {"dims": ("z"), "data": curprof * np.ones_like(_z)},
        "lat": {"dims": ("z"), "data": FWBL_lat[count] * np.ones_like(_z)},
        "lon": {"dims": ("z"), "data": FWBL_lon[count] * np.ones_like(_z)},
        "dist2trackKM": {"dims": ("z"), "data": _dist2trackKM * np.ones_like(_z)},
        "dist2trackRMW": {"dims": ("z"), "data": _dist2trackRMW * np.ones_like(_z)},
        "ild": {"dims": ("z"), "data": ild * np.ones_like(_z)},
        "mld": {"dims": ("z"), "data": mld * np.ones_like(_z)}
    }

    
    # Make the synthetic case
    import copy
    altr_closeIC = copy.deepcopy(closeIC)
    ild_ind = np.where(altr_closeIC['z']['data'] == altr_closeIC['ild']['data'][0])[0][0]
    altr_closeIC['sal']['data'][:ild_ind+1] = altr_closeIC['sal']['data'][ild_ind]

    #########################################################################################################
    # Plot T profile and S profile with ILD and MLD, and time, prof#, distance to track and cloest proximity to track in terms of RMW
    n = 2
    fig = plt.figure(figsize=(12,8))
    gs = fig.add_gridspec(1,n)
    ax = []
    ax = [fig.add_subplot(cell) for cell in gs ];

    y_label_kw = {'labelpad':0}
    ylims = (-100, 0)
    orig_kw = {'color': 'k', 'ls':'-', 'label':'Observed'}#'label':'Barrier Layer'}
    altr_kw = {'color': 'k', 'ls':'--', 'label':'No Barrier Layer'}

    ########################################################
    ## Plot temperature ICs
    xlims =(18,32)
    ax[0].plot(closeIC['temp']['data'],-closeIC['z']['data'], **orig_kw)
    ax[0].plot(altr_closeIC['temp']['data'],-altr_closeIC['z']['data'], **altr_kw)
    ax[0].plot(xlims, (-closeIC['ild']['data'][0],-closeIC['ild']['data'][0]), alpha=0.5, ls=':', c='r', label='ILD',linewidth=2.5)
    ax[0].plot(xlims, (-closeIC['mld']['data'][0],-closeIC['mld']['data'][0]), alpha=0.5, ls=':', c='k', label='MLD',linewidth=2.5)
    # ax[0].annotate('Isothermal Layer Depth', (18,ild+0.4))
    # ax[0].annotate('Mixed Layer Depth', (18,-mld+0.4))

    # Format plot
    ax[0].set_ylabel('Depth [m]', **y_label_kw)
    ax[0].set_xlim(xlims)
    ax[0].set_ylim(ylims)
    ax[0].spines['top']
    ax[0].set_title('Temperature [°C]')
    ax[0].tick_params(labelbottom = False, bottom = False) # Remove bottom tick marks and labels
    ax[0].tick_params(labeltop = True, top = True) # Add top tick marks and labels
    # ax[0].legend()
    # ax[0].grid(axis='y')
    # ax[0].annotate('a) Wind speeds from HRRR at ng645',(time_grid[0,0],ylims[0]),xycoords='data', **ann_kwargs) 
    ########################################################

    ########################################################
    ## Plot salinity ICs
    xlims =(32.4,37)

    ax[1].plot(closeIC['sal']['data'],-closeIC['z']['data'], **orig_kw)
    ax[1].plot(altr_closeIC['sal']['data'],-altr_closeIC['z']['data'], **altr_kw)
    ax[1].plot(xlims, (-closeIC['ild']['data'][0],-closeIC['ild']['data'][0]), alpha=0.5, ls=':', c='r', label='ILD',linewidth=2.5)
    ax[1].plot(xlims, (-closeIC['mld']['data'][0],-closeIC['mld']['data'][0]), alpha=0.5, ls=':', c='k', label='MLD',linewidth=2.5)

    # Format plot
    # ax[1].set_ylabel('Depth [m]', **y_label_kw)
    ax[1].set_xlim(xlims)
    ax[1].set_ylim(ylims)
    ax[1].spines['top']
    ax[1].set_title('Salinity')
    ax[1].tick_params(labelleft = False, left = False) # Remove bottom tick marks and labels
    ax[1].tick_params(labelbottom = False, bottom = False) # Remove bottom tick marks and labels
    ax[1].tick_params(labeltop = True, top = True) # Add top tick marks and labels
    # ax[1].grid(axis='y')
    ax[1].legend(loc='lower left')
    # ax[0].annotate('a) Wind speeds from HRRR at ng645',(time_grid[0,0],ylims[0]),xycoords='data', **ann_kwargs) 
    ########################################################
    plt.savefig(f'{FIGURES_DIR}/'+str(save_prof)+'initialConditionsCloseCase_edit.png',bbox_inches='tight')

    closeIC_ds = xr.Dataset.from_dict(closeIC)
    altr_closeIC_ds = xr.Dataset.from_dict(altr_closeIC)

    fn = f'{OUTPUT_DIR}/ng645-'+str(save_prof)+'_original.nc'
    closeIC_ds.to_netcdf(fn)
    fn = f'{OUTPUT_DIR}/ng645-'+str(save_prof)+'_altered.nc'
    altr_closeIC_ds.to_netcdf(fn)

In [ ]:
proforig19 =xr.open_dataset(f'{OUTPUT_DIR}/ng645-1629339767_original.nc')
profaltr19 = xr.open_dataset(f'{OUTPUT_DIR}/ng645-1629339767_altered.nc')

proforig28=xr.open_dataset(f'{OUTPUT_DIR}/ng645-1630114503_original.nc')
profaltr28=xr.open_dataset(f'{OUTPUT_DIR}/ng645-1630114503_altered.nc')

In [ ]:
# Quick sanity check that the saved initial conditions look right
fig, ax = plt.subplots(1, 2, figsize=(8, 6), sharey=True)

ax[0].plot(proforig19.temp, -proforig19.z, 'k-', lw=2, label='Barrier layer')
ax[0].plot(profaltr19.temp, -profaltr19.z, 'k--', lw=2, label='No barrier layer')
ax[0].set_xlabel('Temperature [°C]')
ax[0].set_ylabel('Depth [m]')
ax[0].set_ylim(-100, 0)

ax[1].plot(proforig19.sal, -proforig19.z, 'k-', lw=2, label='Barrier layer')
ax[1].plot(profaltr19.sal, -profaltr19.z, 'k--', lw=2, label='No barrier layer')
ax[1].set_xlabel('Salinity')
ax[1].legend()

plt.suptitle('Saved initial conditions: 8/19 case')
plt.tight_layout()